In [91]:
import pandas as pd
DATA_PATH = 'jobpostingdata.csv'
MODEL_PATH = 'model.pkl'

df = pd.read_csv(DATA_PATH)

In [92]:
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

In [93]:
def get_tag(tag):
    if tag[0] in ['V', 'N', 'R']:
        return tag[0].lower()
    elif tag[0] == 'J':
        return 'a'
    else:
        return 'n'
    
def lemmatization(tokens):
    lemmatizer = WordNetLemmatizer()
    tagged = pos_tag(tokens)
    lemmatized = []
    for word, tag in tagged:
        result = lemmatizer.lemmatize(word, get_tag(tag))
        lemmatized.append(result)
    return lemmatized

def text_preprocessor(sentence):
    eng_stopwords = stopwords.words('english')
    tokens = word_tokenize(sentence)
    cleaned = [
        token.lower() for token in tokens if token not in eng_stopwords
        and
        token.isalpha()
    ]
    lemmatized = lemmatization(cleaned)
    return lemmatized

In [94]:
def feature_extraction(sentence, existed_words):
    unique_words = set(text_preprocessor(sentence))
    feature = {}
    for word in existed_words:
        feature[word] = (word in unique_words)
    return feature

In [95]:
from nltk.classify.naivebayes import NaiveBayesClassifier
from nltk.classify import accuracy
from sklearn.model_selection import train_test_split

In [96]:
def train_classifier():
    x = df['text']
    y = df['fraudulent']
    corpus = ' '.join(x)
    existed_words = set(text_preprocessor(corpus))

    features = [
        (feature_extraction(sentence, existed_words), category)
        for sentence, category in zip(x, y)
    ]

    train_data, test_data = train_test_split(
        features, test_size=0.2, random_state=42
    )

    classifier = NaiveBayesClassifier.train(train_data)
    acc = accuracy(classifier, test_data) * 100
    print(f"Acc: {acc}")

    return classifier, existed_words

In [97]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [98]:
def train_vectorizer():
    x = df['text']
    vectorizer = TfidfVectorizer(
        ngram_range=(1,2),
        stop_words='english'
    )
    vectors = vectorizer.fit_transform(x)
    return vectorizer, vectors

In [99]:
import os
import pickle

In [100]:
def load_model():
    if os.path.exists(MODEL_PATH):
        with open(MODEL_PATH, 'rb') as f:
            classifier, existed_words, vectorizer, vectors = pickle.load(f)
        return classifier, existed_words, vectorizer, vectors
    else:
        classifier, existed_words = train_classifier()
        vectorizer, vectors = train_vectorizer()

        with open(MODEL_PATH, 'wb') as f:
            pickle.dump((classifier, existed_words, vectorizer, vectors), f)
        return classifier, existed_words, vectorizer, vectors

In [101]:
def print_menu(sentence, category):
    print(f"Sent: {sentence}")
    print(f"Cat: {category}")
    print("1. Write")
    print("2. Recommend")
    print("3. NER")
    print("4. Exit")

    return input("Select choice: ")

In [102]:
def write_sentence():
    sent = ''
    while len(sent.strip()) < 20 or len(sent.strip().split()) < 3:
        sent = input("Input a sentence: ")
    return sent

In [ ]:
def classify_text(classifier, sentence, existed_words):
    feature = feature_extraction(sentence, existed_words)
    category = classifier.classify(feature)
    if category == 1:
        return 'FAKE'
    else:
        return 'REAL'

In [104]:
def load_nlp(nlp, sentence):
    doc = nlp(sentence)
    ents_dicts = {}

    for ent in doc.ents:
        if ent.label_ not in ents_dicts.keys():
            ents_dicts[ent.label_] = []
        ents_dicts[ent.label_].append(doc.text)
        
    return ents_dicts

In [105]:
def print_NER(ents_dict):
    if ents_dict:
        for key, value in ents_dict.items():
            print(f"{key}: {value}")
    else:
        print("NO ENT DICT")

In [106]:
from sklearn.metrics.pairwise import cosine_similarity

In [107]:
def recommend_top_n(vectorizer: TfidfVectorizer, job_vectors, query, topn=5):
    vectorized_query = vectorizer.transform([query])
    similarity = cosine_similarity(job_vectors, vectorized_query).flatten()
    topidx = similarity.argsort()[::-1][:topn]

    for i, idx in enumerate(topidx, 1):
        print(f"{i}. {df['title'].iloc[idx]}")

In [108]:
import spacy

In [115]:
def menu():
    sen = ''
    cat = ''

    classifier = None
    existed_words = None
    vectorizer = None
    vectors = None
    nlp = spacy.load("en_core_web_sm")

    while True:
        choice = print_menu(sen, cat)
        if choice == '1':
            sen = write_sentence()

            if classifier is None or existed_words is None or vectorizer is None or vectors is None:
                classifier, existed_words, vectorizer, vectors = load_model()
            
            category = classify_text(classifier, sen, existed_words)
            print(category)

        elif choice == '2':
            recommend_top_n(vectorizer, vectors, sen)

        elif choice == '3':
            ents_dicts = load_nlp(nlp, sen)
            print_NER(ents_dicts)
            
        elif choice == '4':
            print("Bye My Brother")
            break

In [116]:
menu()

Sent: 
Cat: 
1. Write
2. Recommend
3. NER
4. Exit
TRUE
Sent: i want to break free
Cat: 
1. Write
2. Recommend
3. NER
4. Exit
Bye My Brother
